# Part 3: Sales Forecasting — Advanced Model
**Team:** GenCore | **Lead:** Trịnh Hoàng Tú

**Objective:** Predict daily `Revenue` and `COGS` for 2023-01-01 → 2024-07-01.

**Strategy:**
1. Baseline: Seasonal profile + YoY growth (see `baseline.ipynb`)
2. Advanced: XGBoost with time-based + web traffic features

In [ ]:
# === Environment Detection & Path Config ===
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

IS_KAGGLE = os.path.exists('/kaggle/input')

if IS_KAGGLE:
    DATA_DIR = None
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'sales.csv' in files:
            DATA_DIR = root + '/'
            break
    if DATA_DIR is None:
        raise FileNotFoundError('Could not find sales.csv under /kaggle/input')
    OUT_DIR = '/kaggle/working/'
else:
    DATA_DIR = '../data/raw/'
    OUT_DIR = '../output/'
    sys.path.append(os.path.abspath('..'))

RANDOM_SEED = 42

print(f"Environment: {'Kaggle' if IS_KAGGLE else 'Local'}")
print(f"Data dir:    {DATA_DIR}")

## 1 — Data Loading

In [ ]:
sales = pd.read_csv(os.path.join(DATA_DIR, 'sales.csv'), parse_dates=['Date'])
web_traffic = pd.read_csv(os.path.join(DATA_DIR, 'web_traffic.csv'), parse_dates=['date'])
sample_sub = pd.read_csv(os.path.join(DATA_DIR, 'sample_submission.csv'), parse_dates=['Date'])

print(f"Sales:   {sales.shape} | {sales['Date'].min().date()} → {sales['Date'].max().date()}")
print(f"Test:    {sample_sub.shape} | {sample_sub['Date'].min().date()} → {sample_sub['Date'].max().date()}")
print(f"Traffic: {web_traffic.shape}")
print()
print(sales.head())

## 2 — Feature Engineering
**Anti-leakage:** We only use time features and web traffic. No future Revenue/COGS.

In [ ]:
def create_features(df):
    """Create time-based features from Date column."""
    out = df.copy()
    out['year']       = out['Date'].dt.year
    out['month']      = out['Date'].dt.month
    out['day']        = out['Date'].dt.day
    out['dayofweek']  = out['Date'].dt.dayofweek
    out['dayofyear']  = out['Date'].dt.dayofyear
    out['weekofyear'] = out['Date'].dt.isocalendar().week.astype(int)
    out['is_weekend'] = out['dayofweek'].isin([5, 6]).astype(int)
    # VN context: Tet season, year-end sale
    out['is_tet_season'] = out['month'].isin([1, 2]).astype(int)
    out['is_sale_month'] = out['month'].isin([11, 12]).astype(int)
    return out

def merge_traffic(df, traffic):
    """Left-join web traffic features onto a date-indexed DataFrame."""
    traffic_agg = (
        traffic
        .groupby('date')[['sessions', 'conversion_rate', 'bounce_rate']]
        .mean()
        .reset_index()
    )
    out = df.merge(traffic_agg, left_on='Date', right_on='date', how='left')
    if 'date' in out.columns:
        out.drop(columns=['date'], inplace=True)
    # Forward-fill then back-fill for dates without traffic data
    for col in ['sessions', 'conversion_rate', 'bounce_rate']:
        out[col] = out[col].ffill().bfill()
    return out

In [ ]:
# Build train features
train_df = create_features(sales)
train_df = merge_traffic(train_df, web_traffic)

# Build test features
test_df = create_features(sample_sub[['Date']].copy())
test_df = merge_traffic(test_df, web_traffic)

FEATURE_COLS = [
    'month', 'day', 'dayofweek', 'dayofyear', 'weekofyear',
    'is_weekend', 'is_tet_season', 'is_sale_month',
    'sessions', 'conversion_rate', 'bounce_rate'
]

X_train = train_df[FEATURE_COLS]
X_test  = test_df[FEATURE_COLS]
y_rev   = train_df['Revenue']
y_cogs  = train_df['COGS']

print(f'X_train: {X_train.shape}, X_test: {X_test.shape}')
print(f'y_rev:   {y_rev.shape},   y_cogs: {y_cogs.shape}')
print(f'NaN in X_train: {X_train.isna().sum().sum()}')
print(f'NaN in X_test:  {X_test.isna().sum().sum()}')

## 3 — Train XGBoost

In [ ]:
from xgboost import XGBRegressor

params = dict(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    random_state=RANDOM_SEED,
    n_jobs=-1
)

model_rev = XGBRegressor(**params)
model_rev.fit(X_train, y_rev, verbose=False)

model_cogs = XGBRegressor(**params)
model_cogs.fit(X_train, y_cogs, verbose=False)

print('Models trained successfully.')

## 4 — Evaluate on Training Tail (2021-2022)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# In-sample check on recent years
val_mask = train_df['year'].isin([2021, 2022])
X_val = train_df.loc[val_mask, FEATURE_COLS]
y_val_rev  = train_df.loc[val_mask, 'Revenue']
y_val_cogs = train_df.loc[val_mask, 'COGS']

pred_rev  = model_rev.predict(X_val)
pred_cogs = model_cogs.predict(X_val)

def mape(actual, pred):
    mask = actual != 0
    return (np.abs(actual[mask] - pred[mask]) / actual[mask]).mean() * 100

print('--- Revenue (2021-2022 validation) ---')
print(f'  MAE:  {mean_absolute_error(y_val_rev, pred_rev):,.2f}')
print(f'  RMSE: {np.sqrt(mean_squared_error(y_val_rev, pred_rev)):,.2f}')
print(f'  MAPE: {mape(y_val_rev.values, pred_rev):.2f}%')
print()
print('--- COGS (2021-2022 validation) ---')
print(f'  MAE:  {mean_absolute_error(y_val_cogs, pred_cogs):,.2f}')
print(f'  RMSE: {np.sqrt(mean_squared_error(y_val_cogs, pred_cogs)):,.2f}')
print(f'  MAPE: {mape(y_val_cogs.values, pred_cogs):.2f}%')

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(train_df.loc[val_mask, 'Date'], y_val_rev.values,  lw=0.8, label='Actual')
ax.plot(train_df.loc[val_mask, 'Date'], pred_rev, lw=0.8, ls='--', label='Predicted')
ax.set_title('Revenue — Actual vs Predicted (2021-2022)'); ax.legend()
plt.tight_layout(); plt.show()

## 5 — Feature Importance

In [ ]:
importances = model_rev.feature_importances_
feat_imp = pd.Series(importances, index=FEATURE_COLS).sort_values()
feat_imp.plot.barh(figsize=(8, 5), title='Revenue Model — Feature Importance')
plt.tight_layout(); plt.show()

## 6 — Generate Submission

In [ ]:
rev_pred  = model_rev.predict(X_test)
cogs_pred = model_cogs.predict(X_test)

submission = sample_sub[['Date']].copy()
submission['Revenue'] = np.maximum(0, rev_pred).round(2)
submission['COGS']    = np.maximum(0, cogs_pred).round(2)
submission['Date']    = submission['Date'].dt.strftime('%Y-%m-%d')

os.makedirs(OUT_DIR, exist_ok=True)
submission.to_csv(os.path.join(OUT_DIR, 'submission_tu.csv'), index=False)

print(f'Saved {len(submission)} rows to {OUT_DIR}submission_tu.csv')
submission.head(10)